In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import col, to_timestamp, from_utc_timestamp, date_format, round

In [0]:
jdbc_url = dbutils.secrets.get(scope="Capstone", key="DatabasejdbcUrl")#DatabasejdbcUrl
connection_properties = {
    "user": dbutils.secrets.get(scope="Capstone", key="DatabaseUsername"),
    "password": dbutils.secrets.get(scope="Capstone", key="DatabasePassword"),
    "driver": dbutils.secrets.get(scope="Capstone", key="DatabaseDriver")
}

old_data = spark.read.jdbc(
    url=jdbc_url,
    table="Bronze.Historical_Stock_News",
    properties=connection_properties
)

In [0]:
display(old_data)

In [0]:
expected_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("category", StringType(), True),
    StructField("datetime", TimestampType(), True),
    StructField("headline", StringType(), True),
    StructField("image", StringType(), True),
    StructField("related", StringType(), True),
    StructField("source", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("url", StringType(), True),
    StructField("symbol", StringType(), True)
])

for field in expected_schema:
    if field.name not in old_data.columns:
        raise ValueError(f"Column {field.name} is missing in the old data")

for field in expected_schema:
    if field.dataType == TimestampType():
        old_data = old_data.withColumn(field.name, to_timestamp(col(field.name)))
    else:
        old_data = old_data.withColumn(field.name, col(field.name).cast(field.dataType))

old_data.printSchema()

In [0]:
old_data = old_data.withColumn("datetime", to_timestamp("datetime"))
old_data = old_data.withColumn("datetime", from_utc_timestamp("datetime", "America/New_York"))
old_data = old_data.withColumn("datetime", date_format("datetime", "yyyy-MM-dd HH:mm:ss"))

In [0]:
cleaned_data = old_data.fillna(subset=["category", "headline", "image", "related", "source", "summary", "url", "symbol"], value="None")
# cleaned_data.toPandas().to_csv('/dbfs/FileStore/Silver/Historical_Stock_News.csv', index=False)

In [0]:
start_id = cleaned_data.count() + 1
cleaned_data.toPandas()['id'] = range(start_id, start_id + cleaned_data.count())
cleaned_data.toPandas().to_csv('/dbfs/FileStore/Silver/Historical_Stock_News_Silver.csv', index=False)

In [0]:
display(cleaned_data)

In [0]:
desired_order = [
    'id',
    'category',
    'datetime',
    'headline',
    'image',
    'related',
    'source',
    'summary',
    'url',
    'symbol'
]

stock_news_data_df_silver = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .csv("/FileStore/Silver/Historical_Stock_News_Silver.csv")

stock_news_data_df = stock_news_data_df_silver.select(desired_order)
stock_news_data_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Silver.Historical_Stock_News") \
    .option("user", connection_properties["user"]) \
    .option("password", connection_properties["password"]) \
    .option("driver", connection_properties["driver"]) \
    .mode("overwrite") \
    .option("batchsize", 10000) \
    .option("numPartitions", 8) \
    .save()

print("Data successfully written to Azure SQL Database.")